# PDS 37 with VLTI/PIONIER: H-band data overview

**PIONIER** combines four telescopes in the H band (≈1.65 µm). The data come as a combined OIFITS file where every spectral channel is stored as a separate instrument, which PMOIRED
cannot load (it needs at least two channels per instrument to work out the spectral resolution).
So this notebook reads the tables directly with `astropy` and `pandas` (see `pds37/oifits.py`) and makes
its own plots.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

sys.path.insert(0, str(Path.cwd().parent))
from pds37 import config, oifits

plt.rcParams["figure.dpi"] = 80

vis2 = oifits.read_vis2(config.pionier_file())
t3 = oifits.read_t3(config.pionier_file())
print(f"{len(vis2)} squared visibilities and {len(t3)} closure phases "
      f"over {vis2.night.nunique()} nights")
vis2.groupby("night").agg(n_vis2=("vis2", "size"),
                          max_baseline_m=("baseline_m", "max")).round(1)

## 1. uv coverage
Each point is one baseline (and its mirror image), in units of millions of wavelengths. Longer baselines probe smaller scales.

In [ ]:
nights = sorted(vis2.night.unique())
colors = dict(zip(nights, plt.cm.viridis(np.linspace(0, 1, len(nights)))))

fig, ax = plt.subplots(figsize=(6, 6))
for night, d in vis2.groupby("night"):
    u, v = d.u / d.wl / 1e6, d.v / d.wl / 1e6
    ax.scatter(np.r_[u, -u], np.r_[v, -v], s=12, color=colors[night], label=night)
ax.set(xlim=(120, -120), ylim=(-120, 120), aspect="equal",
       xlabel="u (Mλ, East ←)", ylabel="v (Mλ)", title="PIONIER uv coverage")
ax.legend(fontsize=7, ncol=2);

## 2. Squared visibilities and closure phases

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for night, d in vis2.groupby("night"):
    ax1.errorbar(d.spatial_freq, d.vis2, yerr=d.vis2_err, fmt="o", ms=4, color=colors[night], label=night)
for night, d in t3.groupby("night"):
    ax2.errorbar(d.spatial_freq_max, d.t3phi, yerr=d.t3phi_err, fmt="o", ms=4, color=colors[night])
ax1.set(xlabel="spatial frequency B/λ (Mλ)", ylabel="V²", ylim=(0, 1.15), title="Squared visibility")
ax2.axhline(0, color="k", lw=0.8)
ax2.set(xlabel="longest baseline of triangle / λ (Mλ)", ylabel="closure phase (deg)", title="Closure phase")
ax1.legend(fontsize=7, ncol=2)
fig.tight_layout()

In [ ]:
w = 1 / t3.t3phi_err**2
mean_cp = np.sum(w * t3.t3phi) / np.sum(w)
mean_cp_err = 1 / np.sqrt(np.sum(w))
print(f"Error-weighted mean closure phase: {mean_cp:.2f} ± {mean_cp_err:.2f} deg")

## 3. Characteristic size from a Gaussian fit

As a first, model-agnostic size estimate, V² is fitted with a centred Gaussian plus an unresolved-to-resolved
flux split:

$$V^2(q) = \left[f_\mathrm{c}\,\exp\left(-\frac{(\pi\,\theta\,q)^2}{4\ln 2}\right)\right]^2$$

where $q = B/\lambda$, $\theta$ is the Gaussian FWHM and $f_\mathrm{c}$ the fraction of flux in the compact
component (the rest is assumed fully resolved). This ignores the binary signal, so it is a rough scale, not a physical model.

In [ ]:
MAS_TO_RAD = np.pi / (180 * 3600 * 1000)


def gaussian_vis2(q_mlambda, fwhm_mas, f_compact):
    q = q_mlambda * 1e6
    v = f_compact * np.exp(-(np.pi * fwhm_mas * MAS_TO_RAD * q) ** 2 / (4 * np.log(2)))
    return v**2


popt, pcov = curve_fit(gaussian_vis2, vis2.spatial_freq, vis2.vis2, p0=[1.0, 0.9],
                       sigma=vis2.vis2_err, absolute_sigma=True, bounds=([0, 0], [20, 1]))
perr = np.sqrt(np.diag(pcov))
resid = (vis2.vis2 - gaussian_vis2(vis2.spatial_freq, *popt)) / vis2.vis2_err
red_chi2 = np.sum(resid**2) / (len(vis2) - len(popt))
print(f"FWHM = {popt[0]:.2f} ± {perr[0]:.2f} mas,  compact flux fraction = {popt[1]:.3f} ± {perr[1]:.3f},"
      f"  reduced χ² = {red_chi2:.1f}")

q = np.linspace(0, vis2.spatial_freq.max() * 1.05, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(vis2.spatial_freq, vis2.vis2, yerr=vis2.vis2_err, fmt="o", ms=3, alpha=0.6, label="data")
ax.plot(q, gaussian_vis2(q, *popt), "r-", label=f"Gaussian, FWHM = {popt[0]:.2f} mas")
ax.set(xlabel="spatial frequency (Mλ)", ylabel="V²", ylim=(0, 1.15))
ax.legend();

If the reduced χ² is well above 1, a centred Gaussian does not capture everything: scatter at a given
spatial frequency and non-zero closure phases point to asymmetric structure such as a binary, and a binary fit
to these data would be the natural next step.